# Pareja B · Idea 3 — Cabeza con capas ocultas (MLP)

Una capa lineal solo traza fronteras rectas. Un MLP aprende combinaciones ("tonos oscuros + una figura + ciudad de noche = Misterio"). Rejilla: 1 a 3 capas ocultas, ancho 256/512/1024, ReLU/GELU, dropout 0.2/0.4, lr 1e-3/3e-4, LayerNorm sí/no. Un MLP 512→512→17 tiene ~271 mil parámetros. Empiecen sin esperar a la Pareja A, con los vectores B/32.

**Dónde se programa:** `cabezas.py` (ya tiene `activacion: gelu` y `layernorm: true`; cada capa oculta es Linear → LayerNorm → activación → Dropout). La lógica va en los `.py` (se revisa en el Pull Request); este
notebook solo la corre. **Listo cuando** la mejor configuración está en el registro (se ordena con `python -m clasificador.ranking B3_`).

Antes, una vez para todo el equipo: `00_reproducir_base` (la base `v0.1.0`).


## Preparar el entorno

**CPU basta**: entrenar una cabeza con vectores ya calculados tarda segundos.

- `P2M_DATOS`: dataset compartido (solo lectura). `P2M_SALIDAS`: carpeta **compartida del equipo**
  (`poster2movie_clasificador`), donde quedan vectores, modelos y predicciones para las otras parejas.
  Agreguen las dos a *Mi unidad* como acceso directo.
- Trabaja en la rama **`B-cabezas`** (se crea desde `main` si no existe).

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
os.environ["P2M_DATOS"] = "/content/drive/MyDrive/poster2movie_datos"            # solo lectura
os.environ["P2M_SALIDAS"] = "/content/drive/MyDrive/poster2movie_clasificador"   # compartida del equipo
REPO = "11909/Poster2Movie-Clasificador"
RAMA = "B-cabezas"
DESTINO = "/content/clasificador"
if not os.path.isdir(DESTINO):
    !git clone -q --recurse-submodules https://github.com/{REPO}.git {DESTINO}
%cd {DESTINO}
!git fetch -q origin
!git checkout -q {RAMA} 2>/dev/null || git checkout -q -b {RAMA} origin/main
!git pull -q origin {RAMA} 2>/dev/null; git submodule update -q --init
!pip install -q -r requirements.txt

In [ ]:
# ¿pasan las pruebas de la idea? (GELU, LayerNorm y compatibilidad con los .pt viejos)
!python -m pytest -q tests/test_pareja_b.py -k "mlp or layernorm or cabeza"


## 1. Rejilla

Cada configuración se guarda como `experimentos/B3_<...>.yaml` (se suben todas: aunque no mejore,
se registra). La rejilla completa son **96** configuraciones; con `RAPIDA = True` se corre primero
una **tamizada de 24** (dropout 0.2 y lr 1e-3 fijos: estructura × activación × LayerNorm) y después
se afinan dropout y lr solo en las mejores. Lo que ya se evaluó se salta: se puede volver a correr.


In [ ]:
import itertools, json, os, pathlib

RAPIDA = True                   # False = las 96 configuraciones

OCULTAS = [[256], [512], [1024], [512, 256], [1024, 512], [1024, 512, 256]]
ACTIVACION = ["relu", "gelu"]
LAYERNORM = [False, True]
DROPOUT = [0.2] if RAPIDA else [0.2, 0.4]
TASA = [1e-3] if RAPIDA else [1e-3, 3e-4]


def escribir(ocultas, act, drop, tasa, ln, semilla=None):
    nombre = f"B3_{'x'.join(map(str, ocultas))}_{act}_d{drop}_lr{tasa:g}" + ("_ln" if ln else "")
    cabeza = {"tipo": "mlp", "ocultas": ocultas, "dropout": drop}
    if act != "relu":
        cabeza["activacion"] = act
    if ln:
        cabeza["layernorm"] = True
    if semilla is not None:
        nombre += f"_s{semilla}"
    entrenamiento = {"tasa": tasa, **({"semilla": semilla} if semilla is not None else {})}
    pathlib.Path(f"experimentos/{nombre}.yaml").write_text(
        f"nombre: {nombre}\npareja: B\ndescripcion: Idea 3, rejilla de MLP.\n"
        f"vectores: [clip_b32__centro]\ncabeza: {json.dumps(cabeza)}\n"
        f"entrenamiento: {json.dumps(entrenamiento)}\n")
    return nombre


nombres = [escribir(*c) for c in itertools.product(OCULTAS, ACTIVACION, DROPOUT, TASA, LAYERNORM)]
print(len(nombres), "configuraciones")


In [ ]:
def correr(nombres):
    for nombre in nombres:
        if os.path.exists(f"resultados/{nombre}/metricas_val.json"):
            continue   # ya evaluado
        !python -m clasificador.entrenar experimentos/{nombre}.yaml
        !python -m clasificador.evaluar {nombre}

correr(nombres)


## 2. Las mejores en validación

`ranking` compara contra la base con la regla de publicación (**`cumple`**: el F1 macro no baja y el promedio de los 4 débiles sube) y ordena por el promedio de los débiles. Ojo: con muchas configuraciones, la «mejor» en validación puede serlo por suerte; por eso la sección 4 repite la mejor con otras semillas.


In [ ]:
!python -m clasificador.ranking B3_ --top 10


## 3. Afinar (solo con `RAPIDA = True`)

Toma las 3 mejores de la tamizada y prueba dropout 0.2/0.4 × lr 1e-3/3e-4 con su misma estructura.


In [ ]:
import re, yaml
from clasificador.ranking import ranking

mejores = ranking("B3_").head(3).index
afinar = []
for nombre in mejores:
    c = yaml.safe_load(open(f"experimentos/{nombre}.yaml"))["cabeza"]
    for drop, tasa in itertools.product([0.2, 0.4], [1e-3, 3e-4]):
        afinar.append(escribir(c["ocultas"], c.get("activacion", "relu"), drop, tasa, c.get("layernorm", False)))
afinar = list(dict.fromkeys(afinar))
correr(afinar)
!python -m clasificador.ranking B3_ --top 10


## 4. ¿Es real o es suerte? La mejor, con 3 semillas

Una mejora de 0.002 en F1 macro cabe en el ruido de una semilla. Se repite la mejor con tres semillas: si la diferencia contra la base es mayor que lo que varía entre semillas, es real.


In [ ]:
import pandas as pd

mejor = ranking("B3_").index[0]
c = yaml.safe_load(open(f"experimentos/{mejor}.yaml"))
tasa = c["entrenamiento"]["tasa"]
base = c["cabeza"]
semillas = [escribir(base["ocultas"], base.get("activacion", "relu"), base["dropout"], tasa,
                     base.get("layernorm", False), semilla=s) for s in (1, 2, 3)]
correr(semillas)
registro = pd.read_csv("resultados/registro.csv")
sel = registro[registro["nombre"].isin(semillas + ["base", mejor]) & (registro["split"] == "val")]
sel.drop_duplicates("nombre", keep="last")[["nombre", "F1_macro", "mAP", "F1_Mystery", "F1_History", "F1_Fantasy", "F1_Crime"]]


## Subir a la rama `B-cabezas`

En Colab el repo se borra al cerrar: suban `resultados/` antes. Se sube a **`B-cabezas`**, nunca a
`main`; luego abran el Pull Request (lo revisa alguien de **otra** pareja). Necesitan un token de
GitHub con permiso de escritura (*Settings → Developer settings → Personal access tokens → Tokens (classic)*, con el permiso `repo` (los *fine-grained* no sirven: el repo es de otra cuenta personal)).

In [ ]:
from getpass import getpass
!git config user.name "{input('Tu nombre: ')}"
!git config user.email "{input('Tu correo de GitHub: ')}"
!git add experimentos resultados clasificador tests
!git commit -q -m "{RAMA}: {input('Qué hiciste (mensaje del commit): ')}"
token = getpass("Token de GitHub: ")
!git push -q https://{token}@github.com/{REPO}.git HEAD:{RAMA}
del token
print(f"Abre el Pull Request: https://github.com/{REPO}/compare/main...{RAMA}")